# 🔍 SYSMON EVENTS EXPLORATORY ANALYSIS

This notebook performs comprehensive exploratory analysis on Windows Sysmon events stored in JSONL format from Elasticsearch. The analysis focuses on understanding event distribution, XML structure patterns, field availability, and data quality characteristics.

**Target File**: `ds-logs-windows-sysmon_operational-default-run-01.jsonl`  
**Analysis Type**: 2A-SYSMON  
**Purpose**: Understand Sysmon event structure for optimal CSV conversion strategy

**Key Analysis Areas**:
- Sysmon EventID distribution and frequency patterns
- XML structure analysis and field extraction patterns
- Computer/host distribution analysis
- Temporal patterns and event timeline analysis
- Field availability and completeness assessment
- Data quality and parsing success rates

## Step 0: Import Libraries

In [1]:
import json
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import random
import os
from datetime import datetime
from collections import defaultdict, Counter
import re

## Step 1: The Minimal Unit of Information

Before writing any parsing logic, we need to understand what our raw data looks like. The first step is to read **one single line** from the JSONL file and examine its structure.

In [2]:
# Path to the Sysmon JSONL file — adjust if needed
TARGET_PATH = "../dataset/run-01-apt-1/"
TARGET_FILE = "ds-logs-windows-sysmon_operational-default-run-01.jsonl"
TARGET_FILEPATH = os.path.join(TARGET_PATH, TARGET_FILE)

# Read the first line of the file
with open(TARGET_FILEPATH, 'r') as f:
    first_line = f.readline()

# Parse the JSON string into a Python object
record = json.loads(first_line)

# What type of data structure is it?
print(f"Data type: {type(record)}")
print(f"Number of top-level keys: {len(record)}")

Data type: <class 'dict'>
Number of top-level keys: 16


Each JSONL line is a **dictionary** (Python `dict`) — a single telemetry event with key-value pairs. Let's examine what keys it contains and what type of data each one holds.

## Step 2: Top-Level Structure

Most fields are nested dictionaries (`dict`) — metadata from Elasticsearch and Filebeat (`agent`, `elastic_agent`, `ecs`, `data_stream`, `input`, `tags`). The fields potentially relevant for our research are:

- `@timestamp` — when the event occurred
- `process`, `host`, `user` — partially parsed event data
- `event` — contains event metadata and, crucially, the **original raw event**

Let's look inside the `event` field.

In [3]:
# Explore top-level keys and their data types
for key, value in record.items():
    print(f"{key:20s} -> {type(value).__name__:6s} | {str(value)[:80]}")

agent                -> dict   | {'name': 'WATERFALLS', 'id': '97ceac6e-c74e-4eea-9d56-e6565c1e61b7', 'ephemeral_
process              -> dict   | {'name': 'Microsoft.Exchange.ServiceHost.exe', 'pid': 5864, 'entity_id': '{3fc4f
winlog               -> dict   | {'computer_name': 'WATERFALLS.boombox.local', 'process': {'pid': 3556, 'thread':
log                  -> dict   | {'level': 'information'}
elastic_agent        -> dict   | {'id': '97ceac6e-c74e-4eea-9d56-e6565c1e61b7', 'version': '8.17.3', 'snapshot': 
message              -> str    | Image loaded:
RuleName: -
UtcTime: 2025-03-19 06:09:05.109
ProcessGuid: {3fc4fef
tags                 -> list   | ['preserve_original_event']
input                -> dict   | {'type': 'winlog'}
@timestamp           -> str    | 2025-03-19T06:09:05.109Z
file                 -> dict   | {'path': 'C:\\Windows\\System32\\msvcrt.dll', 'extension': 'dll', 'code_signatur
ecs                  -> dict   | {'version': '8.0.0'}
related              -> dict   | 

## Step 3: The Key Discovery — XML Embedded in JSON

The `event.original` field contains the **raw Sysmon event as an XML string** in the Windows Event Log format. This is the most complete and reliable source of telemetry data — it preserves every field exactly as Sysmon recorded it.

The XML has two main sections:

- **`<System>`** — Event metadata: `EventID` (type of Sysmon event), `Computer` (host name), `TimeCreated`
- **`<EventData>`** — The actual telemetry fields, which **vary by EventID** (e.g., process creation events have `CommandLine`, network events have `SourceIp`)

This means that to extract usable data from these JSONL files, we need an **XML parser** — not just a JSON reader. The next section builds the parsing utilities based on this discovery.

In [4]:
# What's inside the 'event' field?
print(f"Keys in 'event': {list(record['event'].keys())}")
print(f"\nType of event['original']: {type(record['event']['original'])}")
print(f"Length of event['original']: {len(record['event']['original'])} characters")

Keys in 'event': ['agent_id_status', 'ingested', 'original', 'code', 'provider', 'created', 'kind', 'action', 'category', 'type', 'dataset']

Type of event['original']: <class 'str'>
Length of event['original']: 2370 characters


The `event['original']` field contains a long string. Let's see what it looks like:

## Step 4: From Discovery to Parser — Step-by-Step Prototyping

Now that we know the data is XML, we need to extract information programmatically. Let's prototype on the record we already have in memory — starting with Python's built-in `xml.etree.ElementTree`.

In [5]:
# Display the raw content of event['original']
xml_content = record['event']['original']
print(xml_content[:])

<Event xmlns='http://schemas.microsoft.com/win/2004/08/events/event'><System><Provider Name='Microsoft-Windows-Sysmon' Guid='{5770385f-c22a-43e0-bf4c-06f5698ffbd9}'/><EventID>7</EventID><Version>3</Version><Level>4</Level><Task>7</Task><Opcode>0</Opcode><Keywords>0x8000000000000000</Keywords><TimeCreated SystemTime='2025-03-19T06:09:05.866552600Z'/><EventRecordID>11346849</EventRecordID><Correlation/><Execution ProcessID='3556' ThreadID='5560'/><Channel>Microsoft-Windows-Sysmon/Operational</Channel><Computer>WATERFALLS.boombox.local</Computer><Security UserID='S-1-5-18'/></System><EventData><Data Name='RuleName'>-</Data><Data Name='UtcTime'>2025-03-19 06:09:05.109</Data><Data Name='ProcessGuid'>{3fc4fefd-5f81-67da-7700-000000004900}</Data><Data Name='ProcessId'>5864</Data><Data Name='Image'>C:\Program Files\Microsoft\Exchange Server\V15\Bin\Microsoft.Exchange.ServiceHost.exe</Data><Data Name='ImageLoaded'>C:\Windows\System32\msvcrt.dll</Data><Data Name='FileVersion'>7.0.17763.475 (WinB

In [6]:
root = ET.fromstring(xml_content)
print(root.find('EventID'))

None


In [7]:
print(root.find('System'))

None


The `xmlns` attribute declares a default namespace. This means that all elements (`System`, `EventID`, `Computer`, etc.) belong to the namespace `http://schemas.microsoft.com/win/2004/08/events/event`. To find them with `ElementTree`, we must declare the namespace explicitly:

In [8]:
namespaces = {'ns': 'http://schemas.microsoft.com/win/2004/08/events/event'}

# Now we can find elements correctly
root.find('ns:System', namespaces)

<Element '{http://schemas.microsoft.com/win/2004/08/events/event}System' at 0x75591a694400>

With the namespace resolved, we extract the event metadata:

In [9]:
system = root.find('ns:System', namespaces)

event_id_elem = system.find('ns:EventID', namespaces)
computer_elem = system.find('ns:Computer', namespaces)

print(f"EventID:  {event_id_elem.text}")
print(f"Computer: {computer_elem.text}")

EventID:  7
Computer: WATERFALLS.boombox.local


The `<EventData>` section contains multiple `<Data>` elements with a `Name` attribute. We need to iterate over all of them:

In [10]:
event_data = root.find('ns:EventData', namespaces)

fields = {}
for data in event_data.findall('ns:Data', namespaces):
    name = data.get('Name')
    value = data.text
    if name:
        fields[name] = value

# Show all fields for this event (EventID 7 = Image Loaded)
for name, value in fields.items():
    print(f"  {name:20s} = {str(value)[:60]}")

  RuleName             = -
  UtcTime              = 2025-03-19 06:09:05.109
  ProcessGuid          = {3fc4fefd-5f81-67da-7700-000000004900}
  ProcessId            = 5864
  Image                = C:\Program Files\Microsoft\Exchange Server\V15\Bin\Microsoft
  ImageLoaded          = C:\Windows\System32\msvcrt.dll
  FileVersion          = 7.0.17763.475 (WinBuild.160101.0800)
  Description          = Windows NT CRT DLL
  Product              = Microsoft® Windows® Operating System
  Company              = Microsoft Corporation
  OriginalFileName     = msvcrt.dll
  Hashes               = SHA256=39095FE07AC2E244E2180C58BEC2898A0986DDA2BD2ABBC4F739D
  Signed               = true
  Signature            = Microsoft Windows
  SignatureStatus      = Valid
  User                 = NT AUTHORITY\SYSTEM


We have a working prototype for one record. But a JSONL file with 363,657 lines could contain variations in its structure. To analyze the full dataset systematically, we need:

1. **Robust parsing functions** — wrapping our prototype with error handling and XML sanitization
2. **A sampling strategy** — analyzing a representative subset efficiently

The following sections build these components and present the analysis results.

## Step 5: Exploring Variations — Setup

In [11]:
# Analysis Configuration
ANALYSIS_TYPE = "2a-sysmon"
SAMPLE_SIZE = 200_000  # Number of samples to analyze
TARGET_PATH = "../dataset/run-01-apt-1/"
TARGET_FILE = "ds-logs-windows-sysmon_operational-default-run-01.jsonl"
TARGET_FILEPATH = os.path.join(TARGET_PATH, TARGET_FILE)

# Create organized output directory structure
outputs_base_dir = "outputs"
analysis_outputs_dir = f"{outputs_base_dir}/{ANALYSIS_TYPE}"
os.makedirs(analysis_outputs_dir, exist_ok=True)

# Setup logging
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_filename = f"{analysis_outputs_dir}/{ANALYSIS_TYPE}_exploratory_analysis_{timestamp}.log"

def log_print(message):
    """Print and log messages"""
    print(message)
    with open(log_filename, 'a', encoding='utf-8') as f:
        f.write(message + '\n')

# Initialize log file
log_print("SYSMON EVENTS EXPLORATORY ANALYSIS")
log_print(f"Analysis Type: {ANALYSIS_TYPE.upper()}")
log_print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log_print(f"Target File: {TARGET_FILEPATH}")
log_print("=" * 80)
log_print("")

SYSMON EVENTS EXPLORATORY ANALYSIS
Analysis Type: 2A-SYSMON
Generated: 2026-04-21 11:15:52
Target File: ../dataset/run-01-apt-1/ds-logs-windows-sysmon_operational-default-run-01.jsonl



## Step 6: XML Parsing — Robust Functions

Wrapping our prototype from Step 4 into reusable functions with error handling and XML sanitization.

In [12]:
def sanitize_xml(xml_str):
    """Clean invalid characters and repair XML structure"""
    try:
        # Remove non-printable characters
        cleaned = ''.join(c for c in xml_str if 31 < ord(c) < 127 or c in '\t\n\r')
        # Fix common XML issues using BeautifulSoup's parser
        return BeautifulSoup(cleaned, "xml").prettify()
    except:
        return xml_str  # Return original if cleaning fails

def parse_sysmon_event_basic(xml_str):
    """Parse XML to extract basic event information"""
    try:
        # Clean XML first
        clean_xml = sanitize_xml(xml_str)
        
        # Parse with explicit namespace
        namespaces = {'ns': 'http://schemas.microsoft.com/win/2004/08/events/event'}
        root = ET.fromstring(clean_xml)
        
        # System section
        system = root.find('ns:System', namespaces)
        if not system:
            return None, None, None, {}

        event_id_elem = system.find('ns:EventID', namespaces)
        computer_elem = system.find('ns:Computer', namespaces)
        
        event_id = int(event_id_elem.text) if event_id_elem is not None and event_id_elem.text else None
        computer = computer_elem.text if computer_elem is not None and computer_elem.text else None

        # EventData section - extract all fields
        event_data = root.find('ns:EventData', namespaces)
        fields = {}
        if event_data:
            for data in event_data.findall('ns:Data', namespaces):
                name = data.get('Name')
                if name:
                    fields[name] = data.text if data.text else None

        return event_id, computer, len(fields), fields

    except Exception as e:
        return None, None, None, {}

print("✅ XML parsing utilities loaded")

✅ XML parsing utilities loaded


### Dataset Loading and Sampling

With robust parsing utilities in place, we can now load and sample the full dataset.

In [13]:
# Start section logging
log_print("\n" + "=" * 80)
log_print("SECTION 6: DATA LOADING AND SAMPLING")
log_print("=" * 80)
log_print("")

# Count total records
log_print(f"🔄 Counting total records in: {TARGET_FILEPATH}")
total_records = 0
with open(TARGET_FILEPATH, 'r') as f:
    for line in f:
        total_records += 1

log_print(f"📈 Total records in dataset: {total_records:,}")
log_print(f"🎯 Sampling strategy: Will analyze {SAMPLE_SIZE:,} random samples")

# Random sampling strategy
random.seed()  # Use truly random seed for different results each run
sample_indices = set(random.sample(range(total_records), min(SAMPLE_SIZE, total_records)))

log_print(f"📊 Selected {len(sample_indices):,} random indices for analysis")

# End section logging
log_print("\n" + "-" * 60 + " END SECTION " + "-" * 60)
log_print("")


SECTION 6: DATA LOADING AND SAMPLING

🔄 Counting total records in: ../dataset/run-01-apt-1/ds-logs-windows-sysmon_operational-default-run-01.jsonl
📈 Total records in dataset: 363,657
🎯 Sampling strategy: Will analyze 200,000 random samples
📊 Selected 200,000 random indices for analysis

------------------------------------------------------------ END SECTION ------------------------------------------------------------



## Step 7a: Basic Event Structure

In [14]:
# Start section logging
log_print("\n" + "=" * 80)
log_print("SECTION 7: BASIC EVENT STRUCTURE ANALYSIS")
log_print("=" * 80)
log_print("")

# Load and analyze sample data
log_print("📋 BASIC EVENT STRUCTURE ANALYSIS")
log_print("=" * 50)

# Analyze first record structure
sample_data = []
current_index = 0

with open(TARGET_FILEPATH, 'r') as f:
    for line_number, line in enumerate(f):
        if line_number in sample_indices:
            try:
                event = json.loads(line)
                sample_data.append(event)
                if len(sample_data) >= 10:  # Get first 10 for structure analysis
                    break
            except json.JSONDecodeError:
                continue

if sample_data:
    first_sample = sample_data[0]
    log_print(f"🔍 Data type: {type(first_sample)}")
    log_print(f"📏 Number of top-level fields: {len(first_sample)}")
    log_print(f"🗝️  Top-level fields:")
    
    for idx, key in enumerate(first_sample.keys(), 1):
        value_type = type(first_sample[key]).__name__
        log_print(f"   {idx:2d}. {key:30s} ({value_type})")
    
    # Analyze the XML structure
    if 'event' in first_sample and 'original' in first_sample['event']:
        xml_content = first_sample['event']['original']
        log_print(f"\n📄 XML content sample (first 500 chars):")
        log_print("-" * 50)
        log_print(xml_content[:500] + "..." if len(xml_content) > 500 else xml_content)
        
        # Parse the XML to understand structure
        event_id, computer, field_count, fields = parse_sysmon_event_basic(xml_content)
        log_print(f"\n🎯 XML parsing results:")
        log_print(f"   • EventID: {event_id}")
        log_print(f"   • Computer: {computer}")
        log_print(f"   • Field count: {field_count}")
        if fields:
            log_print(f"   • Available fields: {list(fields.keys())[:10]}{'...' if len(fields) > 10 else ''}")
    
    log_print(f"\n📊 Complete first sample structure:")
    log_print("-" * 50)
    log_print(str(first_sample)[:1000] + "..." if len(str(first_sample)) > 1000 else str(first_sample))

# End section logging
log_print("\n" + "-" * 60 + " END SECTION " + "-" * 60)
log_print("")


SECTION 7: BASIC EVENT STRUCTURE ANALYSIS

📋 BASIC EVENT STRUCTURE ANALYSIS
🔍 Data type: <class 'dict'>
📏 Number of top-level fields: 16
🗝️  Top-level fields:
    1. agent                          (dict)
    2. process                        (dict)
    3. winlog                         (dict)
    4. log                            (dict)
    5. elastic_agent                  (dict)
    6. message                        (str)
    7. tags                           (list)
    8. input                          (dict)
    9. @timestamp                     (str)
   10. file                           (dict)
   11. ecs                            (dict)
   12. related                        (dict)
   13. data_stream                    (dict)
   14. host                           (dict)
   15. event                          (dict)
   16. user                           (dict)

📄 XML content sample (first 500 chars):
--------------------------------------------------
<Event xmlns='http://schemas.m

## Step 7b: EventID Distribution

In [15]:
# Start section logging
log_print("\n" + "=" * 80)
log_print("SECTION 8: EVENTID DISTRIBUTION ANALYSIS")
log_print("=" * 80)
log_print("")

log_print("📊 EVENTID DISTRIBUTION ANALYSIS")
log_print("=" * 50)

# Analyze EventID distribution
eventid_counts = Counter()
computer_counts = Counter()
parsing_success = 0
parsing_errors = 0
samples_processed = 0

log_print(f"🔄 Processing {len(sample_indices):,} samples for EventID analysis...")

with open(TARGET_FILEPATH, 'r') as f:
    for line_number, line in enumerate(f):
        if line_number in sample_indices:
            samples_processed += 1
            try:
                event = json.loads(line)
                if 'event' in event and 'original' in event['event']:
                    xml_content = event['event']['original']
                    event_id, computer, field_count, fields = parse_sysmon_event_basic(xml_content)
                    
                    if event_id is not None:
                        eventid_counts[event_id] += 1
                        parsing_success += 1
                        
                        if computer:
                            computer_counts[computer] += 1
                    else:
                        parsing_errors += 1
                else:
                    parsing_errors += 1
                    
            except (json.JSONDecodeError, Exception):
                parsing_errors += 1
            
            # Progress indicator
            if samples_processed % 10000 == 0:
                log_print(f"   Processed {samples_processed:,} samples...")

log_print(f"\n✅ Analysis complete!")
log_print(f"📈 Samples processed: {samples_processed:,}")
log_print(f"📊 Parsing success: {parsing_success:,} ({(parsing_success/samples_processed)*100:.1f}%)")
log_print(f"⚠️  Parsing errors: {parsing_errors:,} ({(parsing_errors/samples_processed)*100:.1f}%)")

# EventID distribution
log_print(f"\n🎯 EVENTID FREQUENCY DISTRIBUTION:")
log_print("Event ID | Count    | Percentage | Description")
log_print("-" * 60)

# Sysmon EventID descriptions for context
eventid_descriptions = {
    1: "Process Creation",
    2: "File Creation Time Changed", 
    3: "Network Connection",
    4: "Sysmon Service State Changed",
    5: "Process Terminated",
    6: "Driver Loaded",
    7: "Image/Library Loaded",
    8: "Create Remote Thread",
    9: "Raw Access Read",
    10: "Process Access",
    11: "File Create",
    12: "Registry Event (Object create/delete)",
    13: "Registry Event (Value Set)",
    14: "Registry Event (Key/Value Rename)",
    15: "File Create Stream Hash",
    16: "Sysmon Config State Changed",
    17: "Pipe Event (Pipe Created)",
    18: "Pipe Event (Pipe Connected)",
    19: "WMI Event (WmiEventFilter activity)",
    20: "WMI Event (WmiEventConsumer activity)",
    21: "WMI Event (WmiEventConsumerToFilter activity)",
    22: "DNS Event (DNS query)",
    23: "File Delete (File Delete archived)",
    24: "Clipboard Change (New content in clipboard)",
    25: "Process Tampering (Process image change)",
    26: "File Delete (File Delete logged)"
}

for event_id, count in eventid_counts.most_common():
    percentage = (count / parsing_success) * 100
    description = eventid_descriptions.get(event_id, "Unknown EventID")
    log_print(f"{event_id:8d} | {count:8,} | {percentage:9.2f}% | {description}")

# Computer distribution
log_print(f"\n🖥️ COMPUTER/HOST DISTRIBUTION:")
log_print(f"📊 Unique computers: {len(computer_counts)}")
log_print("Computer Name        | Count    | Percentage")
log_print("-" * 50)

for computer, count in computer_counts.most_common(10):  # Top 10 computers
    percentage = (count / parsing_success) * 100
    log_print(f"{computer:20s} | {count:8,} | {percentage:9.2f}%")

if len(computer_counts) > 10:
    log_print(f"... and {len(computer_counts) - 10} more computers")

# End section logging
log_print("\n" + "-" * 60 + " END SECTION " + "-" * 60)
log_print("")


SECTION 8: EVENTID DISTRIBUTION ANALYSIS

📊 EVENTID DISTRIBUTION ANALYSIS
🔄 Processing 200,000 samples for EventID analysis...
   Processed 10,000 samples...
   Processed 20,000 samples...
   Processed 30,000 samples...
   Processed 40,000 samples...
   Processed 50,000 samples...
   Processed 60,000 samples...
   Processed 70,000 samples...
   Processed 80,000 samples...
   Processed 90,000 samples...
   Processed 100,000 samples...
   Processed 110,000 samples...
   Processed 120,000 samples...
   Processed 130,000 samples...
   Processed 140,000 samples...
   Processed 150,000 samples...
   Processed 160,000 samples...
   Processed 170,000 samples...
   Processed 180,000 samples...
   Processed 190,000 samples...
   Processed 200,000 samples...

✅ Analysis complete!
📈 Samples processed: 200,000
📊 Parsing success: 199,997 (100.0%)
⚠️  Parsing errors: 3 (0.0%)

🎯 EVENTID FREQUENCY DISTRIBUTION:
Event ID | Count    | Percentage | Description
-------------------------------------------

## Step 7c: Field Availability by EventID

In [16]:
# Start section logging
log_print("\n" + "=" * 80)
log_print("SECTION 9: FIELD AVAILABILITY ANALYSIS BY EVENTID")
log_print("=" * 80)
log_print("")

log_print("📊 FIELD AVAILABILITY ANALYSIS BY EVENTID")
log_print("=" * 50)

# Sysmon field schemas from notebook #2
fields_per_eventid = {
    1: ['UtcTime', 'ProcessGuid', 'ProcessId', 'Image', 'CommandLine', 'CurrentDirectory', 'User', 'Hashes', 'ParentProcessGuid', 'ParentProcessId', 'ParentImage', 'ParentCommandLine'],
    2: ['UtcTime', 'ProcessGuid', 'ProcessId', 'Image', 'TargetFilename', 'CreationUtcTime', 'PreviousCreationUtcTime', 'User'],
    3: ['UtcTime', 'ProcessGuid', 'ProcessId', 'Image', 'User', 'Protocol', 'SourceIsIpv6', 'SourceIp', 'SourceHostname', 'SourcePort', 'SourcePortName', 'DestinationIsIpv6', 'DestinationIp', 'DestinationHostname', 'DestinationPort', 'DestinationPortName'],
    5: ['UtcTime', 'ProcessGuid', 'ProcessId', 'Image', 'User'],
    6: ['UtcTime', 'ImageLoaded', 'Hashes', 'User'],
    7: ['UtcTime', 'ProcessGuid', 'ProcessId', 'Image', 'ImageLoaded', 'OriginalFileName', 'Hashes', 'User'],
    8: ['UtcTime', 'SourceProcessGuid', 'SourceProcessId', 'SourceImage', 'TargetProcessGuid', 'TargetProcessId', 'TargetImage', 'NewThreadId', 'SourceUser', 'TargetUser'],
    9: ['UtcTime', 'ProcessGuid', 'ProcessId', 'Image', 'Device', 'User'],
    10: ['UtcTime', 'SourceProcessGUID', 'SourceProcessId', 'SourceImage', 'TargetProcessGUID', 'TargetProcessId', 'TargetImage', 'SourceThreadId', 'SourceUser', 'TargetUser'],
    11: ['UtcTime', 'ProcessGuid', 'ProcessId', 'Image', 'TargetFilename', 'CreationUtcTime', 'User'],
    12: ['EventType', 'UtcTime', 'ProcessGuid', 'ProcessId', 'Image', 'TargetObject', 'User'],
    13: ['EventType', 'UtcTime', 'ProcessGuid', 'ProcessId', 'Image', 'TargetObject', 'User'],
    15: ['UtcTime', 'ProcessGuid', 'ProcessId', 'Image', 'TargetFilename', 'CreationUtcTime', 'Hash', 'User'],
    17: ['EventType', 'UtcTime', 'ProcessGuid', 'ProcessId', 'PipeName', 'Image', 'User'],
    18: ['EventType', 'UtcTime', 'ProcessGuid', 'ProcessId', 'PipeName', 'Image', 'User'],
    22: ['UtcTime', 'ProcessGuid', 'ProcessId', 'Image', 'QueryName', 'QueryStatus', 'QueryResults', 'User'],
    23: ['UtcTime', 'ProcessGuid', 'ProcessId', 'User', 'Image', 'TargetFilename', 'Hashes'],
    24: ['UtcTime', 'ProcessGuid', 'ProcessId', 'User', 'Image', 'Hashes'],
    25: ['UtcTime', 'ProcessGuid', 'ProcessId', 'User', 'Image']
}

# Analyze field availability for each EventID
field_analysis = {}
samples_per_eventid = {}

log_print(f"🔄 Analyzing field availability across {len(sample_indices):,} samples...")

with open(TARGET_FILEPATH, 'r') as f:
    for line_number, line in enumerate(f):
        if line_number in sample_indices:
            try:
                event = json.loads(line)
                if 'event' in event and 'original' in event['event']:
                    xml_content = event['event']['original']
                    event_id, computer, field_count, fields = parse_sysmon_event_basic(xml_content)
                    
                    if event_id is not None and event_id in fields_per_eventid:
                        if event_id not in field_analysis:
                            field_analysis[event_id] = {}
                            samples_per_eventid[event_id] = 0
                        
                        samples_per_eventid[event_id] += 1
                        
                        # Check each expected field
                        for expected_field in fields_per_eventid[event_id]:
                            if expected_field not in field_analysis[event_id]:
                                field_analysis[event_id][expected_field] = 0
                            
                            if expected_field in fields and fields[expected_field] is not None:
                                field_analysis[event_id][expected_field] += 1
                    
            except (json.JSONDecodeError, Exception):
                continue

# Report field availability
log_print(f"\n📋 FIELD AVAILABILITY REPORT:")
log_print("=" * 60)

for event_id in sorted(field_analysis.keys()):
    total_samples = samples_per_eventid[event_id]
    if total_samples > 0:
        log_print(f"\n🎯 EventID {event_id} ({eventid_descriptions.get(event_id, 'Unknown')})")
        log_print(f"   📊 Analyzed samples: {total_samples:,}")
        log_print(f"   📋 Expected fields: {len(fields_per_eventid[event_id])}")
        log_print("   Field Availability:")
        
        for field in fields_per_eventid[event_id]:
            available_count = field_analysis[event_id].get(field, 0)
            percentage = (available_count / total_samples) * 100
            status = "✅" if percentage > 95 else "⚠️" if percentage > 50 else "❌"
            log_print(f"   {status} {field:25s}: {available_count:6,}/{total_samples:,} ({percentage:5.1f}%)")

# Summary statistics
log_print(f"\n📊 FIELD AVAILABILITY SUMMARY:")
log_print("=" * 40)

total_eventids_analyzed = len(field_analysis)
total_fields_analyzed = sum(len(fields_per_eventid[eid]) for eid in field_analysis.keys())

log_print(f"• Total EventIDs analyzed: {total_eventids_analyzed}")
log_print(f"• Total fields analyzed: {total_fields_analyzed}")
log_print(f"• Field availability varies significantly by EventID")
log_print(f"• Some fields may be conditionally present based on event context")

# End section logging
log_print("\n" + "-" * 60 + " END SECTION " + "-" * 60)
log_print("")


SECTION 9: FIELD AVAILABILITY ANALYSIS BY EVENTID

📊 FIELD AVAILABILITY ANALYSIS BY EVENTID
🔄 Analyzing field availability across 200,000 samples...

📋 FIELD AVAILABILITY REPORT:

🎯 EventID 1 (Process Creation)
   📊 Analyzed samples: 593
   📋 Expected fields: 12
   Field Availability:
   ✅ UtcTime                  :    593/593 (100.0%)
   ✅ ProcessGuid              :    593/593 (100.0%)
   ✅ ProcessId                :    593/593 (100.0%)
   ✅ Image                    :    593/593 (100.0%)
   ✅ CommandLine              :    593/593 (100.0%)
   ✅ CurrentDirectory         :    593/593 (100.0%)
   ✅ User                     :    593/593 (100.0%)
   ✅ Hashes                   :    593/593 (100.0%)
   ✅ ParentProcessGuid        :    593/593 (100.0%)
   ✅ ParentProcessId          :    593/593 (100.0%)
   ✅ ParentImage              :    593/593 (100.0%)
   ✅ ParentCommandLine        :    593/593 (100.0%)

🎯 EventID 2 (File Creation Time Changed)
   📊 Analyzed samples: 118
   📋 Expected fields

## Step 7d: Temporal Patterns

In [17]:
# Start section logging
log_print("\n" + "=" * 80)
log_print("SECTION 10: TEMPORAL PATTERN ANALYSIS")
log_print("=" * 80)
log_print("")

log_print("⏰ TEMPORAL PATTERN ANALYSIS")
log_print("=" * 50)

# Analyze timestamps
timestamps = []
utc_times = []
samples_analyzed = 0

log_print(f"🔄 Extracting temporal data from {len(sample_indices):,} samples...")

with open(TARGET_FILEPATH, 'r') as f:
    for line_number, line in enumerate(f):
        if line_number in sample_indices:
            try:
                event = json.loads(line)
                samples_analyzed += 1
                
                # Extract @timestamp
                if '@timestamp' in event:
                    timestamps.append(event['@timestamp'])
                
                # Extract UtcTime from XML if available
                if 'event' in event and 'original' in event['event']:
                    xml_content = event['event']['original']
                    event_id, computer, field_count, fields = parse_sysmon_event_basic(xml_content)
                    
                    if 'UtcTime' in fields and fields['UtcTime']:
                        utc_times.append(fields['UtcTime'])
                    
            except (json.JSONDecodeError, Exception):
                continue
            
            if samples_analyzed % 50000 == 0:
                log_print(f"   Processed {samples_analyzed:,} samples...")

log_print(f"\n📅 TIMESTAMP ANALYSIS:")
log_print(f"• @timestamp fields found: {len(timestamps):,}")
log_print(f"• UtcTime fields found: {len(utc_times):,}")

if timestamps:
    # Sort timestamps to find range
    sorted_timestamps = sorted(timestamps)
    log_print(f"\n📊 @TIMESTAMP RANGE:")
    log_print(f"• Earliest: {sorted_timestamps[0]}")
    log_print(f"• Latest: {sorted_timestamps[-1]}")
    
    # Sample some timestamps
    log_print(f"\n📄 SAMPLE @TIMESTAMPS:")
    sample_count = min(10, len(sorted_timestamps))
    for i in range(sample_count):
        idx = i * len(sorted_timestamps) // sample_count
        log_print(f"   {i+1:2d}. {sorted_timestamps[idx]}")

if utc_times:
    # Sort UTC times
    sorted_utc = sorted(utc_times)
    log_print(f"\n📊 UTCTIME RANGE:")
    log_print(f"• Earliest: {sorted_utc[0]}")
    log_print(f"• Latest: {sorted_utc[-1]}")
    
    # Sample some UTC times
    log_print(f"\n📄 SAMPLE UTCTIMES:")
    sample_count = min(10, len(sorted_utc))
    for i in range(sample_count):
        idx = i * len(sorted_utc) // sample_count
        log_print(f"   {i+1:2d}. {sorted_utc[idx]}")

# End section logging
log_print("\n" + "-" * 60 + " END SECTION " + "-" * 60)
log_print("")


SECTION 10: TEMPORAL PATTERN ANALYSIS

⏰ TEMPORAL PATTERN ANALYSIS
🔄 Extracting temporal data from 200,000 samples...
   Processed 50,000 samples...
   Processed 100,000 samples...
   Processed 150,000 samples...
   Processed 200,000 samples...

📅 TIMESTAMP ANALYSIS:
• @timestamp fields found: 200,000
• UtcTime fields found: 199,997

📊 @TIMESTAMP RANGE:
• Earliest: 2025-03-19T05:00:00.346Z
• Latest: 2025-03-19T06:12:02.299Z

📄 SAMPLE @TIMESTAMPS:
    1. 2025-03-19T05:00:00.346Z
    2. 2025-03-19T05:05:14.222Z
    3. 2025-03-19T05:08:28.590Z
    4. 2025-03-19T05:12:15.483Z
    5. 2025-03-19T05:21:10.405Z
    6. 2025-03-19T05:34:33.271Z
    7. 2025-03-19T05:39:04.646Z
    8. 2025-03-19T05:42:50.519Z
    9. 2025-03-19T05:58:26.851Z
   10. 2025-03-19T06:01:45.042Z

📊 UTCTIME RANGE:
• Earliest: 2025-03-19 05:00:00.346
• Latest: 2025-03-19 06:12:02.299

📄 SAMPLE UTCTIMES:
    1. 2025-03-19 05:00:00.346
    2. 2025-03-19 05:05:14.222
    3. 2025-03-19 05:08:28.590
    4. 2025-03-19 05:12:15.

## Step 7e: Representative Event Samples

In [18]:
# Start section logging
log_print("\n" + "=" * 80)
log_print("SECTION 11: SAMPLE EVENT DISPLAY")
log_print("=" * 80)
log_print("")

log_print("📄 SAMPLE EVENT DISPLAY")
log_print("=" * 50)

# Display sample events for different EventIDs
sample_events = {}
target_eventids = [1, 3, 7, 12, 13]  # Key Sysmon events to showcase

log_print(f"🔍 Collecting sample events for EventIDs: {target_eventids}")

with open(TARGET_FILEPATH, 'r') as f:
    for line_number, line in enumerate(f):
        if line_number in sample_indices and len(sample_events) < len(target_eventids):
            try:
                event = json.loads(line)
                if 'event' in event and 'original' in event['event']:
                    xml_content = event['event']['original']
                    event_id, computer, field_count, fields = parse_sysmon_event_basic(xml_content)
                    
                    if event_id in target_eventids and event_id not in sample_events:
                        sample_events[event_id] = {
                            'full_event': event,
                            'parsed_fields': fields,
                            'computer': computer,
                            'field_count': field_count
                        }
                        
            except (json.JSONDecodeError, Exception):
                continue

# Display samples
for event_id in sorted(sample_events.keys()):
    sample = sample_events[event_id]
    description = eventid_descriptions.get(event_id, "Unknown")
    
    log_print(f"\n🎯 SAMPLE EVENT - EventID {event_id} ({description})")
    log_print("-" * 60)
    log_print(f"Computer: {sample['computer']}")
    log_print(f"Field Count: {sample['field_count']}")
    log_print(f"@timestamp: {sample['full_event'].get('@timestamp', 'N/A')}")
    
    log_print(f"\nParsed Fields:")
    for field_name, field_value in sample['parsed_fields'].items():
        # Truncate long values for readability
        display_value = str(field_value)[:100] + "..." if field_value and len(str(field_value)) > 100 else field_value
        log_print(f"   • {field_name:20s}: {display_value}")
    
    log_print(f"\nJSON Structure (top-level keys):")
    for key in sample['full_event'].keys():
        value_type = type(sample['full_event'][key]).__name__
        log_print(f"   • {key:20s}: {value_type}")

# End section logging
log_print("\n" + "-" * 60 + " END SECTION " + "-" * 60)
log_print("")


SECTION 11: SAMPLE EVENT DISPLAY

📄 SAMPLE EVENT DISPLAY
🔍 Collecting sample events for EventIDs: [1, 3, 7, 12, 13]

🎯 SAMPLE EVENT - EventID 1 (Process Creation)
------------------------------------------------------------
Computer: WATERFALLS.boombox.local
Field Count: 23
@timestamp: 2025-03-19T06:07:05.801Z

Parsed Fields:
   • RuleName            : -
   • UtcTime             : 2025-03-19 06:07:05.801
   • ProcessGuid         : {3fc4fefd-5f09-67da-2002-000000004800}
   • ProcessId           : 15288
   • Image               : C:\Windows\System32\mmc.exe
   • FileVersion         : 10.0.17763.1697 (WinBuild.160101.0800)
   • Description         : Microsoft Management Console
   • Product             : Microsoft® Windows® Operating System
   • Company             : Microsoft Corporation
   • OriginalFileName    : mmc.exe
   • CommandLine         : "C:\Windows\system32\mmc.exe" "C:\Windows\system32\eventvwr.msc" /s
   • CurrentDirectory    : C:\Windows\system32\
   • User               

## Analysis Conclusions

In [19]:
# Start section logging
log_print("\n" + "=" * 80)
log_print("SECTION 12: ANALYSIS SUMMARY AND RECOMMENDATIONS")
log_print("=" * 80)
log_print("")

log_print("📊 ANALYSIS SUMMARY AND RECOMMENDATIONS")
log_print("=" * 60)

# Calculate summary statistics
total_eventids_found = len(eventid_counts)
most_common_eventid = eventid_counts.most_common(1)[0] if eventid_counts else (None, 0)
parsing_success_rate = (parsing_success / samples_processed) * 100 if samples_processed > 0 else 0

log_print(f"🔍 DATASET CHARACTERISTICS:")
log_print(f"   • Total records in file: ~{total_records:,}")
log_print(f"   • Samples analyzed: {samples_processed:,}")
log_print(f"   • XML parsing success rate: {parsing_success_rate:.1f}%")
log_print(f"   • Unique EventIDs found: {total_eventids_found}")
log_print(f"   • Most common EventID: {most_common_eventid[0]} ({most_common_eventid[1]:,} occurrences)")
log_print(f"   • Unique computers: {len(computer_counts)}")

log_print(f"\n💡 PROCESSING RECOMMENDATIONS:")
log_print(f"   🧠 Memory Management:")
log_print(f"      - Use batch processing for large dataset ({total_records:,} records)")
log_print(f"      - Consider chunked reading for memory efficiency")
log_print(f"      - Implement progress tracking for long operations")

log_print(f"   🛠️  XML Processing:")
log_print(f"      - XML sanitization is crucial for {parsing_errors:,} problematic records")
log_print(f"      - BeautifulSoup XML parser handles malformed XML well")
log_print(f"      - Namespace handling required for proper field extraction")

log_print(f"   📊 EventID-Specific Handling:")
log_print(f"      - Different EventIDs have different field schemas")
log_print(f"      - Field availability varies significantly by EventID")
log_print(f"      - Consider separate processing pipelines per EventID")

log_print(f"   📄 CSV Conversion Strategy:")
log_print(f"      - Use EventID-specific field mappings from schema analysis")
log_print(f"      - Handle missing fields with appropriate default values")
log_print(f"      - Implement robust error logging for malformed XML")
log_print(f"      - Consider unified vs EventID-specific CSV outputs")

log_print(f"\n✅ NEXT STEPS:")
log_print(f"   1. Review findings above")
log_print(f"   2. Update notebook #2 with optimized processing logic")
log_print(f"   3. Implement EventID-specific field validation")
log_print(f"   4. Add comprehensive error handling and logging")
log_print(f"   5. Test with full dataset after validation")

log_print(f"\n🎯 Ready to proceed to notebook #2 optimization!")

# End section logging
log_print("\n" + "-" * 60 + " END SECTION " + "-" * 60)
log_print("")

print(f"\n📋 Analysis complete! Results saved to: {log_filename}")
print(f"📁 Output directory: {analysis_outputs_dir}")
print(f"🎉 Sysmon exploratory analysis complete!")


SECTION 12: ANALYSIS SUMMARY AND RECOMMENDATIONS

📊 ANALYSIS SUMMARY AND RECOMMENDATIONS
🔍 DATASET CHARACTERISTICS:
   • Total records in file: ~363,657
   • Samples analyzed: 200,000
   • XML parsing success rate: 100.0%
   • Unique EventIDs found: 19
   • Most common EventID: 10 (63,106 occurrences)
   • Unique computers: 4

💡 PROCESSING RECOMMENDATIONS:
   🧠 Memory Management:
      - Use batch processing for large dataset (363,657 records)
      - Consider chunked reading for memory efficiency
      - Implement progress tracking for long operations
   🛠️  XML Processing:
      - XML sanitization is crucial for 3 problematic records
      - BeautifulSoup XML parser handles malformed XML well
      - Namespace handling required for proper field extraction
   📊 EventID-Specific Handling:
      - Different EventIDs have different field schemas
      - Field availability varies significantly by EventID
      - Consider separate processing pipelines per EventID
   📄 CSV Conversion Strat